In [0]:

from pyspark.sql.functions import col, sum, avg, count, when, lit

# Load ALL silver tables from supply_chain.silver
silver_customers = spark.table("supply_chain.silver.silver_customers")
silver_drivers = spark.table("supply_chain.silver.silver_drivers")
silver_delivery_events = spark.table("supply_chain.silver.silver_delivery_events")
silver_facilities = spark.table("supply_chain.silver.silver_facilities")
silver_fuel_purchases = spark.table("supply_chain.silver.silver_fuel_purchases")
silver_inventory = spark.table("supply_chain.silver.silver_inventory")
silver_loads = spark.table("supply_chain.silver.silver_loads")
silver_maintenance_records = spark.table("supply_chain.silver.silver_maintenance_records")
silver_routes = spark.table("supply_chain.silver.silver_routes")
silver_safety_incidents = spark.table("supply_chain.silver.silver_safety_incidents")
silver_trailers = spark.table("supply_chain.silver.silver_trailers")
silver_trips = spark.table("supply_chain.silver.silver_trips")
silver_truck_utilization_metrics = spark.table("supply_chain.silver.silver_truck_utilization_metrics")
silver_trucks = spark.table("supply_chain.silver.silver_trucks")
# Load bronze tables (no silver equivalents)
sales_df = spark.table("supply_chain.bronze.sales")
purchase_orders_df = spark.table("supply_chain.bronze.purchase_orders")

# Create mappings for downstream gold transformations
order_items_df = sales_df.select(
    col("sales_id").alias("order_item_id"),
    col("order_id"),
    col("sku_id").alias("product_id"),
    col("location_id").alias("seller_id"),
    col("unit_price").alias("price"),
    lit(0).alias("freight_value"),
    col("quantity_sold"),
    col("total_amount")
)

orders_df = sales_df.select(
    "order_id", 
    "customer_id", 
    "order_date"
).distinct().withColumn(
    "order_status", lit("delivered")
).withColumn(
    "order_delivered_customer_date", col("order_date")
).withColumn(
    "order_estimated_delivery_date", col("order_date")
)

payments_df = sales_df.groupBy("order_id").agg(
    sum("total_amount").alias("payment_value")
)

customers_df = silver_customers

print(f"Loaded {silver_customers.count()} customers")
print(f"Loaded {silver_drivers.count()} drivers")
print(f"Loaded {silver_facilities.count()} facilities")
print(f"Loaded {silver_loads.count()} loads")
print(f"Loaded {silver_inventory.count()} inventory records")
print(f"Loaded {sales_df.count()} sales records")
print(f"Loaded {order_items_df.count()} order items (mapped from sales)")
print(f"Loaded {orders_df.count()} orders (mapped from sales)")

## Inventory snapshot

In [0]:
from pyspark.sql.functions import sum

silver_inventory = spark.table("supply_chain.silver.inventory")

gold_inventory = (
    silver_inventory
    .groupBy("sku_id", "location_id")
    .agg(
        sum("on_hand_quantity").alias("closing_stock"),
        sum("in_transit_qty").alias("in_transit_stock")
    )
)

gold_inventory.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("supply_chain.gold.inventory_snapshot")

## Supplier performance

In [0]:
from pyspark.sql.functions import avg

silver_po = spark.table("supply_chain.silver.silver_purchase_orders")

gold_supplier = (
    silver_po
    .groupBy("sup_id")
    .agg(
        avg("open_qty").alias("avg_open_qty"),
        avg("received_qty").alias("avg_received_qty")
    )
)

gold_supplier.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("supply_chain.gold.supplier_performance")

## LOGISTICS PERFORMANCE

In [0]:
from pyspark.sql.functions import count

silver_loads = spark.table("supply_chain.silver.silver_loads")

gold_logistics = (
    silver_loads
    .groupBy("load_status")
    .agg(
        count("*").alias("total_loads")
    )
)

gold_logistics.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("supply_chain.gold.logistics_performance")

## DEMAND SUPPLY GAP

In [0]:
from pyspark.sql.functions import sum, col

silver_sales = spark.table("supply_chain.silver.silver_sales")
silver_inventory = spark.table("supply_chain.silver.silver_inventory")

# Demand
demand = (
    silver_sales
    .groupBy("sku_id")
    .agg(sum("quantity_sold").alias("total_demand"))
)

# Supply
supply = (
    silver_inventory
    .groupBy("sku_id")
    .agg(sum("on_hand_quantity").alias("total_supply"))
)

# Join
gold_gap = demand.join(
    supply,
    "sku_id",
    "inner"
).withColumn(
    "gap",
    col("total_demand") - col("total_supply")
)

# Save
gold_gap.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("supply_chain.gold.demand_supply_gap")

## PO Fulfillment (Core KPI)

In [0]:
%python
from pyspark.sql.functions import col, sum, avg, count, when

# PO Fulfillment - tracking order items and delivery status
po_fulfillment = (
    order_items_df
    .join(orders_df, "order_id")
    .groupBy("order_id", "seller_id")
    .agg(
        count("order_item_id").alias("total_items"),
        sum("price").alias("total_value"),
        sum("freight_value").alias("total_freight")
    )
    .join(
        orders_df.select("order_id", "order_status", "order_delivered_customer_date", "order_estimated_delivery_date"),
        "order_id"
    )
    .withColumn(
        "is_delivered", 
        when(col("order_status") == "delivered", 1).otherwise(0)
    )
)
po_fulfillment.display()

## Customer Insights

In [0]:
%python
customer_insights = (
    orders_df
    .join(payments_df, "order_id")
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("payment_value").alias("total_spent"),
        avg("payment_value").alias("avg_order_value")
    )
)
print("Customer insights created successfully")
display(customer_insights)

##   Write All Gold Tables

In [0]:
# Create gold-layer schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS supply_chain.`gold-layer`")

# Write all 5 gold tables for dashboard
inventory_snapshot.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
.saveAsTable("supply_chain.`gold-layer`.inventory_snapshot")

po_fulfillment.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
.saveAsTable("supply_chain.`gold-layer`.po_fulfillment")

supplier_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
.saveAsTable("supply_chain.`gold-layer`.supplier_performance")

logistics_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
.saveAsTable("supply_chain.`gold-layer`.logistics_performance")

demand_supply_gap.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
.saveAsTable("supply_chain.`gold-layer`.demand_supply_gap")

print("Successfully created 5 gold tables in supply_chain.gold-layer:")
print("  1. inventory_snapshot - Current stock")
print("  2. po_fulfillment - Procurement KPIs")
print("  3. supplier_performance - OTIF metrics")
print("  4. logistics_performance - Delivery KPIs")
print("  5. demand_supply_gap - Planning insights")

## Optimize

In [0]:
%sql
-- Optimize all 5 gold tables for dashboard
OPTIMIZE supply_chain.`gold-layer`.inventory_snapshot ZORDER BY (sku_id, location_id);
OPTIMIZE supply_chain.`gold-layer`.po_fulfillment ZORDER BY (order_id);
OPTIMIZE supply_chain.`gold-layer`.supplier_performance ZORDER BY (seller_id);
OPTIMIZE supply_chain.`gold-layer`.logistics_performance ZORDER BY (driver_id);
OPTIMIZE supply_chain.`gold-layer`.demand_supply_gap ZORDER BY (product_id); 
